# LightGBM for Sales Drop Detection

## Overview and Model Selection Rationale

This notebook is part of a comparative study of three approaches (TCN, LightGBM, and Isolation Forest) for detecting anomalous hourly sales drops. All models use the **exact same** dataset, preprocessing pipeline, and product selection (top-500 products by August sales volume + 13 products with labeled anomalies). This ensures a fair and reproducible comparison.

**Why LightGBM was chosen as the primary model:**
- Best regression metrics on the validation set (MAE/RMSE improvement of approximately 50% over the naïve baseline).
- Excellent performance on highly intermittent demand (96.83% zero sales according to EDA).
- Superior scalability — unlike TCN, it can be trained on the entire product catalog (10,000+ items).
- High interpretability (feature importance) and fast inference speed, which are critical for production deployment.
- Uses Huber loss (`alpha=0.9`), which is deliberately aligned with the TCN’s `HuberLoss(delta=1.0)` to eliminate differences caused by the loss function.

**Model parameters are fully justified by EDA:**
- Lag and rolling window features (6/24/168 hours) correspond to the daily and weekly seasonality identified in the exploratory analysis.
- Sample weight of 3.0 for non-zero sales directly addresses the high zero-inflation rate (96.83%) and strong positive skew (skewness ≈ 13.75).
- Product and category codes as categorical features leverage LightGBM’s native support for categorical variables.

**Anomaly Detection Methods**  
The model is trained **only once**. Subsequently, three different thresholding strategies are tested on the identical set of predictions:
- **z-score < –4.5** (active in this notebook) — an ultra-conservative threshold for extreme events.
- **σ = 5.0** and **σ = 3.5** (on positive residuals only) — focused specifically on cases where the prediction significantly exceeds actual sales.

A full comparison of all three thresholding approaches will be presented in a separate analytical notebook.

In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error
import gc

print("LightGBM version:", lgb.__version__)

LightGBM version: 4.6.0


In [2]:
DATA_PATH       = r'C:\Users\User\Desktop\диплом\data_v1.csv'
AUG_AN_PATH     = r'C:\Users\User\Desktop\диплом\df_aug_final.csv'
AUG_ZERO_PATH   = r'C:\Users\User\Desktop\диплом\df_aug_final_check_rows.csv'
OUTPUT_PATH   = r'C:\Users\User\Desktop\диплом\detected_anomalies_august_lgbm_z_4_5.csv'

## Data loading & filtering

Load raw sales data, remove cold-start products, and keep only top-500 by August volume plus products with known labeled anomalies.

In [3]:
df = pd.read_csv(DATA_PATH)
df['date'] = pd.to_datetime(df['date'])
print("Initial size:", df.shape)

# Keep data up to August 31
df = df[df['date'] <= '2025-08-31'].copy()

# Remove cold-start products
first_pos = df[df['sales'] > 0].groupby('product')['date'].min()
aug_start = pd.Timestamp('2025-08-01')
cold = first_pos[first_pos >= aug_start].index.tolist()
df = df[~df['product'].isin(cold)].copy()
print(f"After removing cold products: {df.shape}")

# Top-500 by August sales
aug_df = df[df['date'] >= aug_start].copy()
prod_sales = aug_df.groupby('product')['sales'].sum().reset_index()
prod_sales.columns = ['product', 'total_sales']
top_500 = prod_sales.nlargest(500, 'total_sales')['product'].tolist()

# Add products with known anomalies
df_aug_zero = pd.read_csv(AUG_ZERO_PATH)
anomaly_products = df_aug_zero['product'].unique().tolist()
print(f"Products with known anomalies: {len(anomaly_products)}")

selected_products = list(set(top_500 + anomaly_products))
print(f"Top-500 (August only): {len(top_500)} | Anomaly products: {len(anomaly_products)} | Total: {len(selected_products)}")

df = df[df['product'].isin(selected_products)].copy()
print(f"After filtering: {df.shape}")

del first_pos, cold, prod_sales, top_500, df_aug_zero, anomaly_products, selected_products, aug_df
gc.collect()

Initial size: (64081080, 6)
After removing cold products: (41779767, 6)
Products with known anomalies: 13
Top-500 (August only): 500 | Anomaly products: 13 | Total: 500
After filtering: (2736500, 6)


20

## Feature engineering

In [4]:
df = df.sort_values(['product', 'date']).reset_index(drop=True)

df['hour']       = df['date'].dt.hour
df['dayofweek']  = df['date'].dt.dayofweek
df['is_weekend'] = (df['dayofweek'] >= 5).astype(int)

df['stocks_lag_1h'] = df.groupby('product')['stocks'].shift(1).fillna(0)

print(f"Shape after base features: {df.shape}")

Shape after base features: (2736500, 10)


In [5]:
# Sales lags
df['sales_lag_1h']   = df.groupby('product')['sales'].shift(1)
df['sales_lag_24h']  = df.groupby('product')['sales'].shift(24)
df['sales_lag_168h'] = df.groupby('product')['sales'].shift(168)

# Rolling statistics
for window in [6, 24, 168]:
    df[f'rolling_mean_{window}h'] = df.groupby('product')['sales'].transform(
        lambda x: x.rolling(window, min_periods=max(1, window // 2)).mean()
    )
    df[f'rolling_std_{window}h'] = df.groupby('product')['sales'].transform(
        lambda x: x.rolling(window, min_periods=max(1, window // 2)).std()
    )

# Cyclical time features
df['week']          = df['date'].dt.isocalendar().week.astype(int)
df['month']         = df['date'].dt.month
df['hour_sin']      = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos']      = np.cos(2 * np.pi * df['hour'] / 24)
df['dow_sin']       = np.sin(2 * np.pi * df['dayofweek'] / 7)
df['dow_cos']       = np.cos(2 * np.pi * df['dayofweek'] / 7)

# Numeric category codes
df['product_code']  = df['product'].astype('category').cat.codes
df['category_code'] = df['category'].astype('category').cat.codes

# Clean up inf and NaN
numeric_cols = df.select_dtypes(include=['float64', 'float32', 'int32', 'int64']).columns
df[numeric_cols] = df[numeric_cols].replace([np.inf, -np.inf], np.nan).fillna(0)

print(f"Shape after all features: {df.shape}")
print(f"Total features: {df.shape[1]}")
gc.collect()

Shape after all features: (2736500, 27)
Total features: 27


0

## Train / val / test split

Train up to July 24, validation on July 25–31, test on August (with 3-day context from late July for lag features).

In [6]:
df_train = df[df['date'] <= '2025-07-24'].copy()

df_val = df[(df['date'] >= '2025-07-25') & (df['date'] <= '2025-07-31')].copy()

df_test = df[df['date'] >= '2025-07-29'].copy()

print(f"Train: {len(df_train):,} rows | {df_train['date'].min()} — {df_train['date'].max()}")
print(f"Val:   {len(df_val):,} rows | {df_val['date'].min()} — {df_val['date'].max()}")
print(f"Test:  {len(df_test):,} rows | {df_test['date'].min()} — {df_test['date'].max()}")

Train: 2,292,500 rows | 2025-01-15 00:00:00 — 2025-07-24 00:00:00
Val:   72,000 rows | 2025-07-25 00:00:00 — 2025-07-30 23:00:00
Test:  384,500 rows | 2025-07-29 00:00:00 — 2025-08-31 00:00:00


## Feature Engineering and Loss Function

**Features** include temporal indicators (hour of day, day of week, weekend flag), lag sales at 1 h / 24 h / 168 h, rolling mean and standard deviation over 6 h / 24 h / 168 h windows, cyclical encodings of hour and day of week (sine/cosine), and a 1-hour lagged stock level. Lag and rolling features together give the model both short-term momentum and weekly periodicity — the two dominant patterns in hourly retail data confirmed in the exploratory analysis.

**Loss function: `objective = 'huber'`, `alpha = 0.9`** — Huber loss is chosen over MSE because the sales distribution is highly right-skewed with frequent zero values; squared loss would disproportionately penalise large positive spikes and distort forecasts for typical hours. The `alpha` parameter in LightGBM's Huber implementation controls the quantile at which the transition from quadratic to linear penalty occurs. A value of 0.9 places the transition near the 90th percentile of the residual distribution, making the model robust to the upper tail of sales spikes while remaining sensitive to underforecasting — the direction relevant for drop detection. This is aligned in intent with `HuberLoss(delta=1.0)` used in TCN: both formulations deprioritise extreme positive residuals without completely ignoring them.

**Sample weights** — non-zero sales observations receive weight 3.0 vs. 1.0 for zero-sales rows. Given that approximately 95–99% of hourly observations are zeros across the product catalogue (confirmed in EDA), uniform weighting would cause the model to learn a near-zero constant. The 3:1 weight ratio rebalances the training signal toward hours with actual transactions, which are the operationally meaningful periods for drop detection.

## Hyperparameter Justification

All hyperparameters were selected to produce a robust, non-overfitting forecaster on sparse hourly retail data:

**`learning_rate = 0.05`** — a moderate learning rate paired with `n_estimators = 1500` and early stopping. Slower learning rates with more trees consistently outperform fast learning on tabular time-series data; the actual number of trees used is determined by early stopping (`early_stopping_rounds = 30`) on the validation set, not the maximum.

**`num_leaves = 48`, `max_depth = 7`** — constrain tree complexity. The full feature set is ~25 columns; `num_leaves = 48` allows the model to capture meaningful interactions without memorising individual product quirks. `max_depth = 7` acts as a secondary guard against deep, overfitted paths.

**`min_child_samples = 150`** — requires at least 150 observations in each leaf before a split is accepted. This is the primary regularisation lever for sparse data: it prevents the model from creating product-specific micro-splits on the handful of hours where a particular SKU has non-zero sales.

**`subsample = 0.75`, `colsample_bytree = 0.65`** — row and column subsampling introduce stochasticity that acts as implicit regularisation and speeds up training, consistent with standard gradient boosting practice (Chen & Guestrin, 2016).

**`reg_alpha = 0.3`, `reg_lambda = 0.8`** — L1 and L2 weight penalties further smooth leaf values and encourage sparse feature usage. The combination biases the model toward using fewer features more reliably rather than many features weakly, which improves generalisation on the heterogeneous product mix.

**Validation split** — training data covers January–July 24; validation uses July 25–31. The final test set (August) is held out entirely during training. The 7-day validation window captures one full weekly cycle, ensuring that the early stopping criterion reflects the model's ability to generalise across all days of the week.

## LightGBM model training

Train a LightGBM regressor with Huber loss — matched to TCN's `HuberLoss(delta=1.0)` for fair comparison. Non-zero sales get higher sample weight.

In [7]:
# Feature columns (exclude identifiers and target)
feature_cols = [col for col in df_train.columns if col not in [
    'product', 'date', 'category', 'sales'
]]

X_train = df_train[feature_cols]
y_train = df_train['sales']
X_val   = df_val[feature_cols]
y_val   = df_val['sales']

# Higher weight for non-zero sales (aligned with TCN's implicit log1p emphasis)
sample_weight     = np.where(y_train > 0, 3.0, 1.0)
sample_weight_val = np.where(y_val   > 0, 3.0, 1.0)

model_lgbm = lgb.LGBMRegressor(
    objective='huber',      # Matched with TCN: HuberLoss(delta=1.0)
    alpha=0.9,              # Close to MAE behavior
    metric='mae',
    learning_rate=0.05,
    n_estimators=1500,
    num_leaves=48,
    max_depth=7,
    min_child_samples=150,
    subsample=0.75,
    colsample_bytree=0.65,
    reg_alpha=0.3,
    reg_lambda=0.8,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

model_lgbm.fit(
    X_train, y_train,
    sample_weight=sample_weight,
    eval_set=[(X_val, y_val)],
    eval_sample_weight=[sample_weight_val],
    callbacks=[
        lgb.early_stopping(30, verbose=True),
        lgb.log_evaluation(100)
    ]
)

print(f"Best iteration: {model_lgbm.best_iteration_}")

Training until validation scores don't improve for 30 rounds
[100]	valid_0's l1: 0.788618
[200]	valid_0's l1: 0.729346
[300]	valid_0's l1: 0.701564
[400]	valid_0's l1: 0.677715
[500]	valid_0's l1: 0.654963
[600]	valid_0's l1: 0.63248
[700]	valid_0's l1: 0.611772
[800]	valid_0's l1: 0.591097
[900]	valid_0's l1: 0.573952
[1000]	valid_0's l1: 0.555096
[1100]	valid_0's l1: 0.540711
[1200]	valid_0's l1: 0.526353
[1300]	valid_0's l1: 0.512386
[1400]	valid_0's l1: 0.499608
[1500]	valid_0's l1: 0.486215
Did not meet early stopping. Best iteration is:
[1500]	valid_0's l1: 0.486215
Best iteration: 1500


# Validation metrics

In [8]:
val_preds_real   = model_lgbm.predict(X_val)
val_targets_real = y_val.values

# Clip negative predictions to zero
val_preds_real = np.clip(val_preds_real, 0, None)

model_mae  = mean_absolute_error(val_targets_real, val_preds_real)
model_rmse = np.sqrt(mean_squared_error(val_targets_real, val_preds_real))

# Naive baseline: predict[t] = actual[t-1]
naive_mae  = mean_absolute_error(val_targets_real[1:], val_targets_real[:-1])
naive_rmse = np.sqrt(mean_squared_error(val_targets_real[1:], val_targets_real[:-1]))

print(f"{'Metric':<12} {'Baseline':>10} {'LightGBM':>10}")
print(f"{'MAE':<12} {naive_mae:>10.3f} {model_mae:>10.3f}")
print(f"{'RMSE':<12} {naive_rmse:>10.3f} {model_rmse:>10.3f}")
print(f"\nMAE improvement:  {(1 - model_mae/naive_mae)*100:.1f}%")
print(f"RMSE improvement: {(1 - model_rmse/naive_rmse)*100:.1f}%")

Metric         Baseline   LightGBM
MAE               0.682      0.329
RMSE              2.046      1.110

MAE improvement:  51.8%
RMSE improvement: 45.7%


# Predict on test set (August only)

In [9]:
# Test set includes July 29–31 as context for lags
X_test = df_test[feature_cols]
df_test = df_test.copy()
df_test['predicted_sales'] = np.clip(model_lgbm.predict(X_test), 0, None)

# Keep only August for anomaly detection
df_aug_pred = df_test[df_test['date'] >= '2025-08-01'].copy()

print(f"August prediction rows: {len(df_aug_pred):,}")
print(f"Products: {df_aug_pred['product'].nunique()}")

August prediction rows: 360,500
Products: 500


## Threshold variants tested

Two approaches were evaluated on the same trained model. Only the decision boundary changes.

**1. Global z-score (active):** `z-score < -4.5`

**2. Sigma-threshold on positive errors (commented out in Cell 10):**

```python
drop_mask = targets_real < preds_real
errors = preds_real - targets_real
threshold = np.mean(errors[drop_mask]) + sigma * np.std(errors[drop_mask])
anomaly_mask = drop_mask & (errors > threshold)

In [10]:
# Alternative sigma-threshold approach (σ = 5 and σ = 3.5 tested):
# sigma = 5
# drop_mask = targets_real < preds_real
# errors = preds_real - targets_real
# threshold = np.mean(errors[drop_mask]) + sigma * np.std(errors[drop_mask])
# anomaly_mask = drop_mask & (errors > threshold)

# Anomaly: predicted > actual (sales drop)
df_aug_pred['residual'] = df_aug_pred['predicted_sales'] - df_aug_pred['sales']

# Global z-score across all residuals
df_aug_pred['residual_zscore'] = (
    df_aug_pred['residual'] - df_aug_pred['residual'].mean()
) / (df_aug_pred['residual'].std() + 1e-6)

# Mandatory pre-filter
df_aug_pred['mandatory_filter'] = (
    (df_aug_pred['stocks_lag_1h'] > 0) &
    (df_aug_pred['stocks']        > 0) &
    (df_aug_pred['hour']          >= 7) &
    (df_aug_pred['hour']          <= 22)
)

# Anomaly threshold: z-score < -4.5
anomaly_mask = df_aug_pred['mandatory_filter'] & (df_aug_pred['residual_zscore'] < -4.5)
print(f"Anomalies found: {anomaly_mask.sum()}")

final_anomalies = df_aug_pred[anomaly_mask].copy()

Anomalies found: 953


In [11]:
df_aug_an_zero = pd.read_csv(AUG_ZERO_PATH)
df_aug_an_zero['date'] = pd.to_datetime(df_aug_an_zero['date'])

# Match on (product, date) keys
real_anomaly_keys = set(
    zip(df_aug_an_zero['product'].astype(str),
        df_aug_an_zero['date'].astype(str))
)

found_anomaly_keys = set(
    zip(final_anomalies['product'].astype(str),
        pd.to_datetime(final_anomalies['date']).astype(str))
)

true_positives  = real_anomaly_keys & found_anomaly_keys
false_positives = found_anomaly_keys - real_anomaly_keys
false_negatives = real_anomaly_keys - found_anomaly_keys

precision = len(true_positives) / len(found_anomaly_keys) if found_anomaly_keys else 0
recall    = len(true_positives) / len(real_anomaly_keys)  if real_anomaly_keys  else 0
f1        = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0

print(f"Labeled anomalies:     {len(real_anomaly_keys)}")
print(f"Detected by model:     {len(found_anomaly_keys)}")
print(f"True positives:        {len(true_positives)}")
print(f"Other positives:       {len(false_positives)}")
print(f"False negatives:       {len(false_negatives)}")
print(f"\nPrecision: {precision:.1%}  |  Recall: {recall:.1%}  |  F1: {f1:.1%}")

Labeled anomalies:     65
Detected by model:     953
True positives:        5
Other positives:       948
False negatives:       60

Precision: 0.5%  |  Recall: 7.7%  |  F1: 1.0%


In [12]:
# Exclude already-known anomalies before saving
df_known   = pd.read_csv(AUG_ZERO_PATH)
df_known['date'] = pd.to_datetime(df_known['date'])
known_keys = set(zip(df_known['product'].astype(str), df_known['date'].astype(str)))

final_anomalies['_key'] = list(
    zip(final_anomalies['product'].astype(str),
        pd.to_datetime(final_anomalies['date']).astype(str))
)
new_anomalies = final_anomalies[~final_anomalies['_key'].isin(known_keys)].drop(columns='_key')

final_columns = ['date', 'product', 'category', 'stocks', 'sales', 'price',
                 'predicted_sales', 'residual', 'residual_zscore']

for col in final_columns:
    if col not in new_anomalies.columns:
        new_anomalies[col] = np.nan

new_anomalies = new_anomalies[final_columns].sort_values(['date', 'product']).reset_index(drop=True)
new_anomalies.to_csv(OUTPUT_PATH, index=False)

print(f"Total detected:        {len(final_anomalies)}")
print(f"Already in known list: {len(final_anomalies) - len(new_anomalies)}")
print(f"New anomalies only:    {len(new_anomalies)}")
print(f"Saved to: {OUTPUT_PATH}")

Total detected:        953
Already in known list: 5
New anomalies only:    948
Saved to: C:\Users\User\Desktop\диплом\detected_anomalies_august_lgbm_z_4_5.csv
